# Notebook 05: Thresholds, model choice, freeze and 2025 scoring

**Study:** predicting recorded road casualty severity (KSI) in Great Britain. This notebook carries out sections 3.7.3 and 3.7.4 of the methodology, up to the point where the 2025 outcomes would be joined.

**What it does**
1. **Thresholds (3.7.3).** For each of the nine model and training combinations it takes the pooled out-of-fold scores and finds two thresholds: the 0.5 reference and the one that maximises pooled macro-F1 on a grid from 0.05 to 0.95 in steps of 0.01 (ties: closest to 0.5, then the lower). That gives 18 sets of decisions.
2. **Model choice (3.7.3).** Ranks the nine combinations by mean fold macro-F1 at the macro-F1 threshold and applies the one-standard-error rule (prefer logistic regression, then random forest, then boosting; within a family the higher mean, ties in favour of untreated, weighting, oversampling).
3. **Freeze (3.7.4).** Writes the freeze record **before** any refit or 2025 scoring: checksums of every input, the chosen settings and thresholds, and the selected model.
4. **Refit and score (3.7.4).** Refits all nine combinations on all 2023-2024 records, scores the 2025 casualties **from features only**, and applies both thresholds. No recalibration on 2025 data.

**The 2025 labels are not opened.** The notebook never reads the sealed labels file. Notebook 06 does that, after the freeze record and 2025 scores exist.

**Dry run:** set `QUICK_TEST = True` for a few-minute check on the pilot rows. It writes to a separate folder.

In [1]:
# 1. Setup -----------------------------------------------------------------
import os, sys, json, hashlib, subprocess, time, gc, datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
try:
    import imblearn, psutil
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'imbalanced-learn', 'psutil'], check=True)
    import imblearn, psutil
try:
    display
except NameError:
    display = print

QUICK_TEST = False          # True = dry run on pilot rows and 5,000 test rows, separate output folder
MAX_RUN_MINUTES = 0         # 0 = no limit; otherwise stop cleanly between fits and rerun to continue

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/stats19_severity')
else:
    PROJECT = Path(os.environ.get('PROJECT_DIR', './stats19_severity'))
CODE = PROJECT / 'code'; MOD = PROJECT / 'outputs' / 'models'; ENV = PROJECT / 'environment'
DATA = PROJECT / 'data' / 'model_inputs'; SEALED = PROJECT / 'data' / 'sealed_2025_labels'
SEARCH = MOD / ('search_quicktest' if QUICK_TEST else 'search')
FINAL = MOD / ('final_quicktest' if QUICK_TEST else 'final')
SCORES = FINAL / 'scores_2025'
for p in (FINAL, SCORES): p.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(CODE))
from stats19_lib import *

SEED = 20261107
N_JOBS = os.cpu_count()
T_START = time.time()
(ENV / 'pip_freeze_notebook05.txt').write_text(subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout)
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def read_frame(stem):
    for ext, fn in (('.parquet', pd.read_parquet), ('.pkl', pd.read_pickle)):
        if Path(str(stem) + ext).exists(): return fn(str(stem) + ext), Path(str(stem) + ext)
    raise FileNotFoundError(stem)
print('scikit-learn', sklearn.__version__, '| cores', N_JOBS, '| QUICK_TEST', QUICK_TEST)

Mounted at /content/drive
scikit-learn 1.6.1 | cores 2 | QUICK_TEST False


In [2]:
# 2. Load development data, out-of-fold scores, selected settings -----------------------
assert sha(DATA / 'folds_dev.csv') == json.loads((MOD / 'folds_checksum.json').read_text())['sha256'], 'folds file changed'
dev, NUM, CAT = load_dev(PROJECT)
oof, oof_path = read_frame(SEARCH / 'oof_scores_dev')
if QUICK_TEST: dev = dev[dev['pilot'] == 1]
dev = dev.reset_index(drop=True)
assert (oof['cas_key'].values == dev['cas_key'].values).all(), 'out-of-fold scores are not aligned with the development rows'
y = dev['ksi'].values.astype(int); fold = dev['fold'].values
COMBOS = [(k, t) for k in KINDS for t in TREATMENTS]
selected = json.loads((SEARCH / 'selected_settings.json').read_text())
print('Development rows:', len(dev), '| KSI rate', round(y.mean(), 4))
print('Selected settings (from notebook 04):')
for k in KINDS: print(f"  {k}: cfg {selected[k]['cfg']} {selected[k]['params']}")

Development rows: 261249 | KSI rate 0.2117
Selected settings (from notebook 04):
  lr: cfg 4 {'C': 0.1}
  rf: cfg 1 {'max_features': 0.2, 'min_samples_leaf': 10, 'min_samples_split': 10}
  hgb: cfg 2 {'learning_rate': 0.05, 'max_iter': 200, 'max_leaf_nodes': 63, 'min_samples_leaf': 50, 'l2_regularization': 0.0}


In [3]:
# 3. Thresholds: 0.5 reference and the pooled macro-F1 maximiser (3.7.3) ----------------------
GRID = np.round(np.arange(0.05, 0.95 + 1e-9, 0.01), 2)
assert len(GRID) == 91
def counts(y, p, thresholds):
    out = []
    pos = y == 1
    for t in thresholds:
        flag = p >= t
        tp = int((flag & pos).sum()); fp = int((flag & ~pos).sum()); fn = int((~flag & pos).sum()); tn = int((~flag & ~pos).sum())
        out.append((tp, fp, fn, tn))
    return np.array(out)
def macro_f1(c):
    tp, fp, fn, tn = c[..., 0], c[..., 1], c[..., 2], c[..., 3]
    f_pos = np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
    f_neg = np.where(2 * tn + fn + fp > 0, 2 * tn / np.maximum(2 * tn + fn + fp, 1), 0.0)
    return (f_pos + f_neg) / 2
def describe(c):
    tp, fp, fn, tn = c
    rec = tp / (tp + fn); spec = tn / (tn + fp)
    return dict(tp=tp, fp=fp, fn=fn, tn=tn, recall=rec, precision=tp / (tp + fp) if tp + fp else np.nan, specificity=spec,
                balanced_accuracy=(rec + spec) / 2, flagged_share=(tp + fp) / (tp + fp + fn + tn))

rows, THR = [], {}
for k, t in COMBOS:
    p = oof[f'{k}__{t}'].values
    cc = counts(y, p, GRID); m = macro_f1(cc)
    best = m.max()
    cand = GRID[np.abs(m - best) <= 1e-12]
    opt = sorted(cand, key=lambda x: (abs(x - 0.5), x))[0]          # closest to 0.5, then the lower one
    THR[(k, t)] = dict(ref=0.5, opt=float(opt))
    for name, thr in (('reference 0.5', 0.5), ('macro-F1 maximiser', opt)):
        c = counts(y, p, [thr])[0]
        rows.append(dict(model=k, training=t, rule=name, threshold=round(float(thr), 2), macro_f1=float(macro_f1(c)), n_tied_optima=len(cand) if name != 'reference 0.5' else np.nan, **describe(c)))
thr_df = pd.DataFrame(rows)
thr_df.round(4).to_csv(FINAL / 'thresholds_dev.csv', index=False)
print('Pooled out-of-fold decisions: 18 sets (nine combinations, two thresholds)')
display(thr_df[['model', 'training', 'rule', 'threshold', 'macro_f1', 'recall', 'precision', 'specificity', 'flagged_share']].round(3))

Pooled out-of-fold decisions: 18 sets (nine combinations, two thresholds)


,model,training,rule,threshold,macro_f1,recall,precision,specificity,flagged_share
0,lr,untreated,reference 0.5,0.50,0.556,0.141,0.598,0.975,0.050
1,lr,untreated,macro-F1 maximiser,0.31,0.641,0.450,0.428,0.838,0.223
2,lr,weight,reference 0.5,0.50,0.605,0.698,0.346,0.646,0.427
3,lr,weight,macro-F1 maximiser,0.62,0.641,0.463,0.421,0.829,0.232
4,lr,oversample,reference 0.5,0.50,0.605,0.697,0.346,0.646,0.426
5,lr,oversample,macro-F1 maximiser,0.62,0.641,0.463,0.421,0.829,0.233
6,rf,untreated,reference 0.5,0.50,0.554,0.135,0.628,0.978,0.046
7,rf,untreated,macro-F1 maximiser,0.30,0.648,0.483,0.429,0.828,0.238
8,rf,weight,reference 0.5,0.50,0.647,0.523,0.415,0.802,0.267
9,rf,weight,macro-F1 maximiser,0.52,0.648,0.479,0.430,0.830,0.236


In [4]:
# 4. Model choice: mean fold macro-F1 and the one-standard-error rule (3.7.3) -----------------
fold_rows = []
for k, t in COMBOS:
    p = oof[f'{k}__{t}'].values
    for f in sorted(np.unique(fold)):
        m = fold == f
        fold_rows.append(dict(model=k, training=t, fold=int(f), macro_f1_opt=float(macro_f1(counts(y[m], p[m], [THR[(k, t)]['opt']])[0])),
                              macro_f1_ref=float(macro_f1(counts(y[m], p[m], [0.5])[0]))))
fm = pd.DataFrame(fold_rows)
rank = fm.groupby(['model', 'training']).agg(mean=('macro_f1_opt', 'mean'), sd=('macro_f1_opt', lambda s: s.std(ddof=1)),
                                           mean_at_ref=('macro_f1_ref', 'mean')).reset_index()
rank['threshold'] = [THR[(a, b)]['opt'] for a, b in zip(rank['model'], rank['training'])]
rank = rank.sort_values('mean', ascending=False).reset_index(drop=True)
leader = rank.iloc[0]
n_folds = fm['fold'].nunique()
SE = leader['sd'] / np.sqrt(n_folds)
rank['within_margin'] = rank['mean'] >= leader['mean'] - SE
display(rank.round(4))
print(f"Leader: {leader['model']} / {leader['training']} (mean macro-F1 {leader['mean']:.4f}); one standard error = {SE:.4f}; margin = {leader['mean'] - SE:.4f}")

FAMILY_ORDER, TREAT_ORDER = ['lr', 'rf', 'hgb'], ['untreated', 'weight', 'oversample']
inside = rank[rank['within_margin']]
family = next(f for f in FAMILY_ORDER if (inside['model'] == f).any())
fam = inside[inside['model'] == family].copy()
top = fam['mean'].max()
fam = fam[(top - fam['mean']).abs() < 1e-9]                      # exact ties only
chosen = sorted(fam['training'], key=TREAT_ORDER.index)[0]
SELECTED = dict(model=family, training=chosen, mean_fold_macro_f1=float(rank[(rank['model'] == family) & (rank['training'] == chosen)]['mean'].iloc[0]),
                leader=f"{leader['model']}/{leader['training']}", one_se=float(SE))
print(f"\nSelected for the predictor analysis: {family} with {chosen} training.")
if (family, chosen) != (leader['model'], leader['training']):
    print('This is not the top-ranked combination: it is inside the one-standard-error margin and preferred as simpler.')
rank.round(5).to_csv(FINAL / 'model_ranking_dev.csv', index=False)
fm.round(5).to_csv(FINAL / 'fold_macro_f1_dev.csv', index=False)
(FINAL / 'selected_for_importance.json').write_text(json.dumps(SELECTED, indent=1))
print('Fold macro-F1 values are optimistic: the thresholds were tuned on the same pooled scores. They rank candidates and are not performance estimates.')

,model,training,mean,sd,mean_at_ref,threshold,within_margin
0,hgb,untreated,0.6509,0.0040,0.5673,0.30,True
1,hgb,weight,0.6504,0.0036,0.6161,0.62,True
2,hgb,oversample,0.6500,0.0032,0.6157,0.62,True
3,rf,untreated,0.6481,0.0032,0.5539,0.30,False
4,rf,weight,0.6477,0.0033,0.6467,0.52,False
5,rf,oversample,0.6471,0.0038,0.6460,0.51,False
6,lr,untreated,0.6414,0.0043,0.5562,0.31,False
7,lr,weight,0.6408,0.0043,0.6054,0.62,False
8,lr,oversample,0.6406,0.0041,0.6055,0.62,False


Leader: hgb / untreated (mean macro-F1 0.6509); one standard error = 0.0018; margin = 0.6491

Selected for the predictor analysis: hgb with untreated training.
Fold macro-F1 values are optimistic: the thresholds were tuned on the same pooled scores. They rank candidates and are not performance estimates.


In [5]:
# 5. Freeze record, written BEFORE any refit or 2025 scoring --------------------------------
thr_json = {f'{k}__{t}': v for (k, t), v in THR.items()}
inputs = {
    'lib': sha(CODE / 'stats19_lib.py'),
    'features_dev': sha(read_frame(DATA / 'features_dev_2023_2024')[1]),
    'features_test_2025_no_labels': sha(read_frame(DATA / 'features_test_2025_NO_LABELS')[1]),
    'folds_dev': sha(DATA / 'folds_dev.csv'),
    'oof_scores_dev': sha(oof_path),
    'selected_settings': sha(SEARCH / 'selected_settings.json'),
    'search_summary': sha(SEARCH / 'search_summary.csv'),
    'search_configs_round2': sha(SEARCH / 'search_configs_round2.json') if (SEARCH / 'search_configs_round2.json').exists() else None,
    'thresholds_dev': sha(FINAL / 'thresholds_dev.csv'),
    'model_ranking_dev': sha(FINAL / 'model_ranking_dev.csv'),
}
sealed_file = SEALED / 'labels_test_2025_SEALED.csv'
prep = json.loads((PROJECT / 'outputs' / 'features' / 'preprocessing_summary.json').read_text())
record = dict(
    created=dt.datetime.now().isoformat(timespec='seconds'), purpose='Freeze before refit and 2025 scoring (methodology 3.7.4)',
    seed=SEED, sklearn=sklearn.__version__, rare_min_freq=RARE_MIN_FREQ, rf_trees=RF_TREES,
    predictors=dict(numeric=NUM, categorical=CAT), settings=selected, thresholds=thr_json, selected_for_importance=SELECTED,
    input_sha256=inputs,
    sealed_labels=dict(sha256_recorded_in_notebook02=prep['sealed_labels_sha256'], file_bytes=sealed_file.stat().st_size, read_by_this_notebook=False),
    labels_read_before_freeze=False)
rec_path = FINAL / 'freeze_record.json'
if rec_path.exists():
    old = json.loads(rec_path.read_text())
    same = old['input_sha256'] == record['input_sha256'] and old['thresholds'] == record['thresholds'] and old['settings'] == record['settings']
    assert same, 'The freeze record already exists and the inputs, settings or thresholds now differ. Do not overwrite it: investigate and log the change.'
    print('Freeze record already exists and matches the current inputs:', old['created'])
else:
    rec_path.write_text(json.dumps(record, indent=1, default=str))
    print('Freeze record written:', rec_path)
print(json.dumps({k: v for k, v in record.items() if k not in ('predictors', 'settings', 'thresholds', 'input_sha256')}, indent=1, default=str))

Freeze record written: /content/drive/MyDrive/stats19_severity/outputs/models/final/freeze_record.json
{
 "created": "2026-11-07T17:23:20",
 "purpose": "Freeze before refit and 2025 scoring (methodology 3.7.4)",
 "seed": 20261107,
 "sklearn": "1.6.1",
 "rare_min_freq": 100,
 "rf_trees": 100,
 "selected_for_importance": {
  "model": "hgb",
  "training": "untreated",
  "mean_fold_macro_f1": 0.650864305880157,
  "leader": "hgb/untreated",
  "one_se": 0.0018111588106018669
 },
 "sealed_labels": {
  "sha256_recorded_in_notebook02": "22ddae8fde110307daf9bd711a84f0b9cd18166d24e52194082b60e53cc3d4e3",
  "file_bytes": 2558009,
  "read_by_this_notebook": false
 },
 "labels_read_before_freeze": false
}


In [6]:
# 6. Refit all nine combinations on the full development set and score 2025 (features only) -----------
test, _ = read_frame(DATA / 'features_test_2025_NO_LABELS')
assert 'ksi' not in test.columns and set(test.columns) == set(['cas_key'] + NUM + CAT), 'the 2025 feature file must hold features only'
if QUICK_TEST: test = test.head(5000)
test = test.reset_index(drop=True)
print('2025 rows to score:', len(test), '| development rows for the refit:', len(dev))
class OutOfTime(Exception): pass
log_rows = []
try:
    for kind in KINDS:
        for trt in TREATMENTS:
            f = SCORES / f'{kind}__{trt}.npy'
            if f.exists(): continue
            if MAX_RUN_MINUTES and (time.time() - T_START) / 60 > MAX_RUN_MINUTES: raise OutOfTime()
            p, info = fit_predict(kind, selected[kind]['params'], trt, dev, y, test, NUM, CAT, SEED, N_JOBS)
            assert len(p) == len(test) and np.isfinite(p).all()
            tmp = f.with_suffix('.tmp.npy'); np.save(tmp, p.astype(np.float32)); tmp.replace(f)
            print(f"{kind:>3} {trt:<10} refit on {info['n_train']} rows | {(info['t_prep'] + info['t_fit'] + info['t_pred']) / 60:.1f} min | elapsed {(time.time() - T_START) / 3600:.2f} h")
            gc.collect()
except OutOfTime:
    print(f'Stopped after {MAX_RUN_MINUTES} minutes. Rerun this cell to continue; finished combinations are kept.')
    raise SystemExit
print('All nine refits finished.')

2025 rows to score: 127883 | development rows for the refit: 261249
 lr untreated  refit on 261249 rows | 0.3 min | elapsed 0.02 h
 lr weight     refit on 261249 rows | 0.3 min | elapsed 0.02 h
 lr oversample refit on 411906 rows | 0.5 min | elapsed 0.03 h
 rf untreated  refit on 261249 rows | 4.6 min | elapsed 0.11 h
 rf weight     refit on 261249 rows | 4.5 min | elapsed 0.18 h
 rf oversample refit on 411906 rows | 8.0 min | elapsed 0.31 h
hgb untreated  refit on 261249 rows | 3.7 min | elapsed 0.37 h
hgb weight     refit on 261249 rows | 3.7 min | elapsed 0.44 h
hgb oversample refit on 411906 rows | 5.5 min | elapsed 0.53 h
All nine refits finished.


In [7]:
# 7. Apply both thresholds, check, and seal the scores ------------------------------------------------
out = test[['cas_key']].copy()
rows = []
for k, t in COMBOS:
    p = np.load(SCORES / f'{k}__{t}.npy').astype(float)
    assert ((p >= 0) & (p <= 1)).all()
    col = f'{k}__{t}'
    out[col] = p
    out[col + '__ref'] = (p >= THR[(k, t)]['ref']).astype(np.int8)
    out[col + '__opt'] = (p >= THR[(k, t)]['opt']).astype(np.int8)
    dev_p = oof[col].values
    rows.append(dict(model=k, training=t, thr_opt=THR[(k, t)]['opt'], mean_score_2025=p.mean(), mean_score_dev_oof=dev_p.mean(),
                     flagged_ref_2025=out[col + '__ref'].mean(), flagged_ref_dev=(dev_p >= 0.5).mean(),
                     flagged_opt_2025=out[col + '__opt'].mean(), flagged_opt_dev=(dev_p >= THR[(k, t)]['opt']).mean()))
desc = pd.DataFrame(rows)
display(desc.round(3))
print('These compare score levels and flagged shares on 2025 with the development scores. No labels are involved.')
assert out['cas_key'].is_unique
strata25 = pd.read_csv(DATA / 'eval_strata_2025.csv', usecols=['cas_key'])
if not QUICK_TEST:
    assert set(out['cas_key']) == set(strata25['cas_key']), '2025 scores do not cover the 2025 casualties'
desc.round(4).to_csv(FINAL / 'scores_2025_description.csv', index=False)
score_file = FINAL / 'test_scores_and_decisions_2025.parquet'
try:
    out.to_parquet(score_file, index=False)
except Exception:
    score_file = FINAL / 'test_scores_and_decisions_2025.pkl'; out.to_pickle(score_file)
rec = json.loads(rec_path.read_text())
rec['scores_2025'] = dict(file=score_file.name, sha256=sha(score_file), rows=int(len(out)), written=dt.datetime.now().isoformat(timespec='seconds'),
                          labels_read_before_scores=False, recalibrated_on_2025=False)
rec_path.write_text(json.dumps(rec, indent=1, default=str))
print('\nSaved:', score_file.name, '| sha256', rec['scores_2025']['sha256'][:16])
print('Freeze record complete. The 2025 labels have not been opened.')
print('NEXT: notebook 06 joins the sealed labels and computes the evaluation measures (Table 3.3) with the paired collision bootstrap.')

,model,training,thr_opt,mean_score_2025,mean_score_dev_oof,flagged_ref_2025,flagged_ref_dev,flagged_opt_2025,flagged_opt_dev
0,lr,untreated,0.31,0.221,0.212,0.058,0.050,0.244,0.223
1,lr,weight,0.62,0.463,0.449,0.450,0.427,0.253,0.232
2,lr,oversample,0.62,0.463,0.449,0.451,0.426,0.255,0.233
3,rf,untreated,0.30,0.224,0.213,0.054,0.046,0.258,0.238
4,rf,weight,0.52,0.388,0.372,0.286,0.267,0.254,0.236
5,rf,oversample,0.51,0.384,0.368,0.278,0.258,0.262,0.242
6,hgb,untreated,0.30,0.222,0.211,0.059,0.053,0.257,0.236
7,hgb,weight,0.62,0.456,0.439,0.444,0.417,0.242,0.220
8,hgb,oversample,0.62,0.455,0.437,0.444,0.415,0.240,0.217


These compare score levels and flagged shares on 2025 with the development scores. No labels are involved.

Saved: test_scores_and_decisions_2025.parquet | sha256 010c53a0a7f95b36
Freeze record complete. The 2025 labels have not been opened.
NEXT: notebook 06 joins the sealed labels and computes the evaluation measures (Table 3.3) with the paired collision bootstrap.
